# Module 3: Prefill, Decode, and the KV Cache

In Module 2 you sized the KV cache. This module builds it. You work self-attention by hand in numpy, then measure the real thing on your own server. By the end you can explain the trade the KV cache makes: spend GPU memory to avoid recomputing the past.

## Learning objectives
- Compute self-attention by hand and see why the scores matrix is quadratic in sequence length
- Separate prefill from decode and explain why decode is sequential
- Compare naive generation against KV-cached generation and measure the difference
- Measure time to first token and tokens per second on your own server
- Resolve vLLM metric names at runtime instead of hardcoding them
- Watch the live KV-cache gauge move while a request runs, and see prefix caching skip work

## Prerequisites
- Finished Module 2
- Your vLLM Deployment serves `Qwen/Qwen3-4B` and is reachable for the live cells
- About 18 minutes

References: [Attention is all you need](https://arxiv.org/abs/1706.03762) &middot; [vLLM PagedAttention](https://docs.vllm.ai/en/latest/design/kv_cache.html) &middot; [vLLM automatic prefix caching](https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/)

## Prefill and decode design basics

A request runs in two phases, and they cost different things.

- Prefill reads the whole prompt in one pass. Every prompt token is processed together, so prefill is parallel across the prompt's tokens and fast per token. It fills the KV cache with the keys and values for the prompt. The time to the first generated token is the prefill latency, also called time to first token.
- Decode generates the answer one token at a time. Each step reads all the model weights and the growing KV cache to produce the next token, appends that token's key and value to the cache, and repeats. Decode is sequential. The per-token cost is the time per output token.

![One request: prefill once, then decode one token at a time while the KV cache grows](images/03_prefill_decode_architecture.png)

The KV cache is the trade at the center of this module. Without it, every decode step would recompute the keys and values for the entire sequence so far. With it, you store them once and reuse them.

## 1. Setup

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31" "numpy>=1.26"

In [ ]:
import sys, time, threading
from pathlib import Path

if Path("../common").exists():
    REPO_ROOT = Path("..")
else:
    REPO_ROOT = Path(".")
sys.path.insert(0, str(REPO_ROOT.resolve()))

import numpy as np
from common.config import get_settings, print_settings, build_client
from common import foundation

settings = get_settings()
settings.model_name = foundation.served_model_name(settings)  # use the model the server actually serves
print_settings(settings)
client = build_client(settings)

**What you should see:** your resolved settings. The numpy sections run locally with no GPU; the measurement sections need your live endpoint.

## 2. Self-attention by hand

Attention is the operation the KV cache exists to serve, so compute it once by hand. For a sequence of tokens, each token builds a query, a key, and a value. The attention scores are every query against every key, which forms a square matrix: one row and one column per token. That square is the cost. It grows with the square of the sequence length.

In [ ]:
# One causal attention head on a short toy sequence, in plain numpy.
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

rng = np.random.default_rng(0)
T, d = 6, 8                       # 6 tokens, head dimension 8
X = rng.standard_normal((T, d))   # toy token embeddings
Wq, Wk, Wv = (rng.standard_normal((d, d)) for _ in range(3))

Q, K, V = X @ Wq, X @ Wk, X @ Wv
scores = Q @ K.T / np.sqrt(d)                      # T x T: every query against every key
causal = np.triu(np.ones((T, T), bool), k=1)       # a token cannot attend to the future
scores = np.where(causal, -np.inf, scores)
weights = softmax(scores, axis=-1)
out = weights @ V

print("scores matrix shape:", scores.shape, "-> T x T, quadratic in sequence length")
print("each row sums to 1 after softmax:", np.allclose(weights.sum(axis=1), 1))
print("attention output shape:", out.shape, "(one vector per token)")

**What you should see:** a 6 by 6 scores matrix. Double the tokens and that matrix quadruples. That quadratic is why long context is expensive. The keys and values you keep so each new token can attend back are linear in length; the scores stay quadratic. The KV cache stores those linear keys and values so each step skips recomputing them.

## 3. The two phases

Prefill computes all T rows of the scores matrix you just built at once, in one pass, and stores every token's key and value. Decode then adds one new row per step: it computes the new token's query, attends over every stored key and value, picks the next token, and appends that token's key and value. The cache is one token longer each step. That one-new-row-at-a-time dependency is why decode is sequential, and why its speed is bounded by how fast the GPU reads the weights, the ceiling you computed in Module 2. Prefill is parallel across the prompt, so it is fast per token.

## 4. Naive against KV-cached generation

Make the trade concrete. Generate a sequence two ways with the same weights and the same tokens. Use a fixed sequence and skip token sampling so the only thing that varies is the key/value work each step does; that per-step work is identical to real decode. Naive generation recomputes the keys and values for the entire prefix at every step. Cached generation keeps them and computes only the new token's key and value. Count the work and time both, then confirm they produce the same answer.

In [ ]:
# Same single-head attention, two generation strategies, identical weights and tokens.
d = 32
Wq = rng.standard_normal((d, d)); Wk = rng.standard_normal((d, d)); Wv = rng.standard_normal((d, d))
N = 300
seq = rng.standard_normal((N, d))   # one fixed token sequence, shared by both runs

def attend(Q, Kc, Vc):              # Q is one query row; attend over all cached keys/values
    s = Q @ Kc.T / np.sqrt(d)
    return softmax(s, -1) @ Vc

def naive_generate(X):
    last = None
    for t in range(1, len(X) + 1):
        Xt = X[:t]
        K, V = Xt @ Wk, Xt @ Wv     # recompute K, V for the WHOLE prefix every step
        last = attend((Xt @ Wq)[-1:], K, V)
    return last

def cached_generate(X):
    Kc = np.empty((0, d)); Vc = np.empty((0, d)); last = None
    for t in range(len(X)):
        xt = X[t:t+1]
        Kc = np.vstack([Kc, xt @ Wk]); Vc = np.vstack([Vc, xt @ Wv])   # only the new token
        last = attend(xt @ Wq, Kc, Vc)
    return last

naive_kv = sum(range(1, N + 1))     # N(N+1)/2
cached_kv = N

t0 = time.time(); a = naive_generate(seq); t_naive = time.time() - t0
t0 = time.time(); b = cached_generate(seq); t_cached = time.time() - t0

print(f"naive  K/V computes: {naive_kv:,}  (= N(N+1)/2, recompute the prefix each step)")
print(f"cached K/V computes: {cached_kv:,}  (= N, only the new token)")
print(f"reduction: {naive_kv / cached_kv:.0f}x fewer K/V computes for N={N}")
print(f"wall time: naive {t_naive*1000:.0f} ms vs cached {t_cached*1000:.0f} ms ({t_naive/t_cached:.1f}x)")
print(f"same answer: {np.allclose(a, b)}")

**What you should see:** the cached path doing roughly N/2 times less key/value work (150x at N=300, and the gap widens linearly as N grows) and producing an identical answer. The wall-clock gap is smaller than the compute-count gap at this size because numpy is fast on small matrices and the cached path pays a little to grow the cache, but the ratio widens with length. This is exactly what the KV cache buys on the GPU: it turns the quadratic recompute into linear work, at the cost of the memory you sized in Module 2.

## 5. Measure time to first token

Now measure the real thing on your server. Stream one request and time the first token (prefill) and the gaps between tokens (decode). Then read the server's own time-to-first-token histogram. vLLM has renamed metrics across versions, so resolve the metric name at runtime rather than hardcoding it.

In [ ]:
# Requires a live vLLM endpoint. Client-side TTFT and decode rate for one streamed request.
result = foundation.stream_and_time(
    client, settings.model_name,
    [{"role": "user", "content": "Explain why GPU memory bandwidth limits decode speed, in three sentences."}],
    max_tokens=128,
)
ttft_ms = result["ttft_s"] * 1000 if result["ttft_s"] else None
tpot_ms = result["tpot_s"] * 1000 if result["tpot_s"] else None
print(f"time to first token (prefill) : {ttft_ms:.0f} ms" if ttft_ms else "no tokens received")
if tpot_ms:
    print(f"time per output token (decode): {tpot_ms:.1f} ms  ({1000 / tpot_ms:.0f} tokens/s)")
print(f"streamed {result['tokens']} tokens")

**What you should see:** a TTFT in the low hundreds of milliseconds and a decode rate of tens of tokens per second, in line with the ceiling from Module 2. Network adds a little to the client-side number, which is why you cross-check against the server's own histogram next.

In [ ]:
# Requires a live vLLM endpoint. Read the server's TTFT histogram, resolving the name at runtime.
m = foundation.fetch_metrics(settings)
ttft_name = m.resolve("ttft")
print("server exposes TTFT under:", ttft_name)
server_ttft = m.histogram_avg("ttft")
if server_ttft is not None:
    print(f"server average TTFT since start: {server_ttft * 1000:.0f} ms")
else:
    print("no TTFT observations yet; send a few requests first")

**What you should see:** the real metric name the server emits (`vllm:time_to_first_token_seconds` on this build) and the server-side average TTFT. This average covers every request since the server started, so it will not exactly match the single request above; it is the server's own view rather than the client's. Resolving the name at runtime is the discipline that keeps these notebooks working across vLLM versions. You read this endpoint in every load module.

## 6. Watch the KV cache fill

The KV cache is not an abstraction. The server reports its usage as a gauge you can read while a request runs. Send a long request in the background and sample the gauge: it sits at zero when idle, climbs as the request decodes and its cache grows, and drops back when the request finishes and its blocks are freed.

In [ ]:
# Requires a live vLLM endpoint. Sample the KV-cache usage gauge while a long request runs.
def long_request():
    client.chat.completions.create(
        model=settings.model_name,
        messages=[{"role": "user", "content": "Write a detailed 400-word explanation of how a GPU serves an LLM."}],
        max_tokens=512, temperature=0.0,
    )

print(f"idle KV usage: {foundation.read_gauge('kv_cache_usage', settings) or 0.0:.3f}")
worker = threading.Thread(target=long_request, daemon=True)
worker.start()

peak = 0.0
while worker.is_alive():
    usage = foundation.read_gauge("kv_cache_usage", settings) or 0.0
    peak = max(peak, usage)
    time.sleep(0.3)
worker.join()
print(f"peak KV usage while running: {peak:.3f}")
print(f"KV usage after finish: {foundation.read_gauge('kv_cache_usage', settings) or 0.0:.3f}")

**What you should see:** usage at or near zero when idle, a higher peak while the request decodes, and a return toward zero after it completes. One request uses a small fraction of the pool. Module 7 fills this gauge under concurrent load and shows what happens when it reaches the top.

## 7. Cold against warm: prefix caching

vLLM hashes each block by its token IDs and the hash of the block before it, so a request that begins with the same tokens maps to the same blocks. Those leading blocks are reused and their prefill is skipped. Show it: send a request with a long unique prefix (cold, nothing to reuse), then send a second request with the same prefix and a different question (warm, the prefix is cached). Read the prefix-cache counters and compare time to first token.

This demo needs prefix caching enabled on the server, which is the default here. Confirm it with `foundation.kv_cache_info(settings)["enable_prefix_caching"]`. If it is off, both requests show a 0% hit rate and no speedup.

In [ ]:
# Requires a live vLLM endpoint. A long, unique shared prefix so the first request is a true miss.
import uuid
tag = uuid.uuid4().hex[:8]
shared_prefix = (f"Reference document {tag}. " +
                 "GPU memory bandwidth bounds decode because each token rereads the weights. " * 120)

def ask(question):
    msgs = [{"role": "user", "content": shared_prefix + "\n\nQuestion: " + question}]
    before = foundation.fetch_metrics(settings)
    q0 = before.gauge("prefix_cache_queries") or 0
    h0 = before.gauge("prefix_cache_hits") or 0
    r = foundation.stream_and_time(client, settings.model_name, msgs, max_tokens=32)
    after = foundation.fetch_metrics(settings)
    dq = (after.gauge("prefix_cache_queries") or 0) - q0
    dh = (after.gauge("prefix_cache_hits") or 0) - h0
    hit_rate = (dh / dq) if dq else 0.0
    return r["ttft_s"] * 1000 if r["ttft_s"] else None, hit_rate

cold_ttft, cold_hit = ask("Summarize the reference in one sentence.")
warm_ttft, warm_hit = ask("List two facts from the reference.")
print(f"cold: TTFT {cold_ttft:.0f} ms, prefix hit rate {cold_hit:.0%}")
print(f"warm: TTFT {warm_ttft:.0f} ms, prefix hit rate {warm_hit:.0%}")
if cold_ttft and warm_ttft:
    print(f"warm is {cold_ttft / warm_ttft:.1f}x faster to first token")

**What you should see:** the cold request with a low prefix-cache hit rate and a higher TTFT, and the warm request with a high hit rate and a much lower TTFT, because the shared prefix was not re-prefilled. For an agent with a fixed system prompt and tool schemas, or a RAG app with a shared document, this is free latency: structure the shared text as a common prefix and every request after the first skips it. That is the per-step prefill cut promised in Module 1.

> NOTE: On a busy shared server other traffic can evict your blocks between the two calls, which lowers the warm hit rate. Run the two cells back to back for the cleanest result.

## Things to know

- **Prefill is parallel across the prompt's tokens, decode is sequential.** Time to first token is prefill; time per output token is decode. Decode is the slow phase and the one bounded by weight bandwidth.
- **The scores matrix is quadratic, the KV cache is linear.** The cache trades memory for not recomputing the past, turning quadratic recompute into linear work.
- **Resolve metric names at runtime.** vLLM renames gauges across versions. Ask the server what it exposes rather than hardcoding a name.
- **The KV gauge is real and live.** You can watch one request fill it. Under load it is the first signal of saturation.
- **Prefix caching is free latency for shared text.** A fixed system prompt or a shared document is prefilled once and reused.

## Try it yourself

**Grow the sequence.** Set `N = 600` in section 4 and re-run. How does the naive-versus-cached wall-clock ratio change as the sequence gets longer?

**Vary the prompt length.** In section 5, send a very short prompt and a very long one. Predict which has the higher TTFT and by roughly what factor before you run it, then measure. Cross-check the long prompt against the server histogram from the same section.

**Break the cache.** In section 7, change one word near the start of `shared_prefix` between the cold and warm calls. The hit rate should fall, because the blocks no longer hash the same. **Stretch:** change a word near the end instead and see how much still hits.

## Summary

- You computed self-attention by hand and saw the quadratic scores matrix.
- You separated prefill from decode and explained why decode is sequential.
- You compared naive and KV-cached generation and measured the reduction in work.
- You measured time to first token and tokens per second on your server, resolving the metric name at runtime.
- You watched the KV cache fill during a request and saw a cached prefix come back faster.

## Next

**Module 4: Dense versus MoE on the GPU.** You know why decode is bounded by reading the weights. Next you put that on the roofline, see the memory-bound and compute-bound regions and their crossover, and learn how a mixture-of-experts model generates faster than its total size suggests. That is the first "read fewer bytes" idea, and it hands off to Omer's optimization block.